# 02 · Data acquisition

**Phase skill: pulling data from a REST API** — request parameters, pagination,
politeness, and caching raw data to disk.

The repo's pipeline already ingests Open5e *monsters*. You will pull a dataset the
pipeline has never touched: the **spells** endpoint, `https://api.open5e.com/v1/spells/`,
filtered to the open-content 5e SRD document. No code in `src/` answers these exercises.

Ground rules for talking to someone else's server:

- always pass a timeout,
- pause between requests (this workbook uses ≥ 0.2 s),
- download once, save raw, and work from the file after that.

In [ ]:
import json
import time
from pathlib import Path

import requests

BASE_URL = "https://api.open5e.com/v1/spells/"
DOC_SLUG = "wotc-srd"   # the open-content 5e SRD document
PAGE_SIZE = 100

RAW_DIR = Path("data") / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = RAW_DIR / "open5e_spells.json"
print(f"Raw file will land at {RAW_PATH.resolve()}")

## Exercise 2.1 — fetch one page

GET one page from `BASE_URL`, filtered to the SRD document and with a page size of 100.
Both the filter and the page size travel as **query parameters**: the API filters on
`document__slug`, and sizes pages with `limit`.

**Produce:** `page` — the response parsed from JSON into a Python `dict`.

<details><summary>Hint 1 (concept)</summary>

One "page" is a single HTTP GET. The filters are not part of the URL path — they ride along as query parameters, and the library builds the `?a=b&c=d` tail for you.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `requests.get(url, params=..., timeout=...)` and the `.json()` method on the response.

</details>

In [ ]:
# Produce: page (dict) -- first page of SRD spells, page size 100

page = ...

type(page)

In [ ]:
assert isinstance(page, dict), f"page should be a dict parsed from JSON, got {type(page).__name__}"
for key in ("count", "next", "results"):
    assert key in page, f"page has no {key!r} key -- is this the parsed body of the right endpoint?"
assert isinstance(page["results"], list), "page['results'] should be a list of spell records"
assert len(page["results"]) == 100, (
    f"first page holds {len(page['results'])} records, expected 100 -- check the page-size parameter")
first = page["results"][0]
assert first.get("document__slug") == "wotc-srd", (
    f"first record belongs to document {first.get('document__slug')!r}, not the SRD -- check the filter parameter")
print("PASS: one page fetched -- 100 SRD spell records.")
print("Context: APIs page big collections; the fields around 'results' are metadata telling you how much more exists.")

## Exercise 2.2 — how big is the whole collection?

You downloaded 100 records, but the response *metadata* states the size of the entire
filtered collection without downloading it.

**Produce:** `total_count` — that number, as an `int`, read from `page` (no new request).

<details><summary>Hint 1 (concept)</summary>

The page you already have describes the whole result set, not just its own slice.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Print `page.keys()` and look at what the non-`results` fields hold.

</details>

In [ ]:
# Produce: total_count (int) -- size of the full filtered collection, from page metadata

total_count = ...

total_count

In [ ]:
assert isinstance(total_count, int), f"total_count should be an int, got {type(total_count).__name__}"
assert 287 <= total_count <= 351, (
    f"total_count = {total_count}, outside 287-351. The API held ~319 SRD spells when this "
    "workbook was built; small drift is fine, this much means a wrong field or filter")
print(f"PASS: the API reports {total_count} SRD spells in total.")
print("Context: count metadata lets you plan the download -- and later, verify you got everything.")

## Exercise 2.3 — loop all pages, politely

Download **every** SRD spell record into one list. Each page tells you where the next
page is; stop when there isn't one. Pause at least 0.2 s between requests — you are a
guest on someone's server.

**Produce:** `all_spells` — a list of all record dicts, in the order the API returned them.

<details><summary>Hint 1 (concept)</summary>

Pagination is a loop with a moving target: each response hands you the URL of the next request, and the last page hands you nothing.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `while` loops over `page.get("next")`, and `time.sleep`.

</details>

In [ ]:
# Produce: all_spells (list of dicts) -- every SRD spell record

all_spells = []

...

print(f"downloaded {len(all_spells)} spells")

In [ ]:
assert len(all_spells) == total_count, (
    f"all_spells holds {len(all_spells)} records but the API promised {total_count} -- "
    "a page was skipped, duplicated, or the loop stopped early")
slugs = {s["slug"] for s in all_spells}
assert len(slugs) == len(all_spells), (
    f"{len(all_spells) - len(slugs)} duplicate records -- was the same page appended twice?")
for key in ("name", "level_int", "school"):
    assert key in all_spells[0], f"records lack the {key!r} field -- did something other than result records get appended?"
print(f"PASS: all {len(all_spells)} spells downloaded, no duplicates.")
print("Context: polite pagination -- small pauses, no hammering -- is the difference between a client and a scraper that gets blocked.")

## Exercise 2.4 — save raw, then cache

Two steps:

1. Save `all_spells` to `RAW_PATH` as JSON. Raw data on disk is your reproducibility
   insurance — the API can change or vanish; your analysis shouldn't.
2. Write a function `load_spells_cached()` returning a tuple `(spells, from_cache)`:
   if `RAW_PATH` exists, load from disk and return `from_cache=True`; otherwise fetch
   all pages (as in 2.3), save, and return `from_cache=False`.

**Produce:** the saved file, the function, and `spells, from_cache = load_spells_cached()`.

<details><summary>Hint 1 (concept)</summary>

A cache is just a check: is the expensive thing already on disk? The function should be safe to call any number of times and hit the network at most once, ever.

</details>
<details><summary>Hint 2 (what to look up)</summary>

Look up `Path.exists`, `json.dump` / `json.load` (or `Path.write_text` / `read_text` with `json.dumps` / `loads`).

</details>

In [ ]:
# Step 1: save all_spells to RAW_PATH as JSON
...

# Step 2: the cache
def load_spells_cached():
    """Return (spells, from_cache) -- load RAW_PATH if it exists, else fetch and save."""
    ...

spells, from_cache = load_spells_cached()
print(len(spells), from_cache)

In [ ]:
assert RAW_PATH.exists(), f"{RAW_PATH} does not exist -- step 1 saves the raw download to disk"
on_disk = json.loads(RAW_PATH.read_text())
assert len(on_disk) == len(all_spells), (
    f"the file holds {len(on_disk)} records but you downloaded {len(all_spells)}")
spells2, cached2 = load_spells_cached()
assert cached2 is True, "the file exists, so a fresh call should report from_cache=True without touching the network"
assert len(spells2) == len(all_spells), f"the cache returned {len(spells2)} records, expected {len(all_spells)}"
print(f"PASS: {RAW_PATH} saved ({len(on_disk)} records) and the cache skips the network.")
print("Context: notebook 03 reads this exact file -- acquisition ends when the raw data is safely on disk.")

**Done.** `03_store.ipynb` picks up `data/raw/open5e_spells.json` from here and turns it
into a queryable table.